In [14]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os
# Params
VERBOSE = True
CHOSEN = 'qwen'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged','7-wonders']
IT = 5
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/'
OVERWRITE = ['']
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/'
    N_GPU_LAYERS = -1
except:
    #LOCAL
    BASE_FOLDER = './'
    N_GPU_LAYERS = 20
RULE_FOLDER = os.path.join(BASE_FOLDER, 'rules/texts/')
PROMPT_FOLDER = os.path.join(BASE_FOLDER, 'prompts/')
OUT_FOLDER = os.path.join(BASE_FOLDER, 'extraction/')

In [15]:


# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q6_K.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"bartowski/Qwen2.5-7B-Instruct-GGUF",
             'filename': "Qwen2.5-7B-Instruct-Q6_K.gguf",
             'temperature': 0.6,
             'n_ctx': 32768,
             'chat_format': "qwen"},
    'gemma': {'repo_id':"bartowski/google_gemma-3n-E4B-it-GGUF",
                'filename':"google_gemma-3n-E4B-it-Q6_K.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': None
             },
}
model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=N_GPU_LAYERS, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=VERBOSE,
                            force_download=True,
                            enable_thinking=True)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce GTX 1070, compute capability 6.1, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce GTX 1070) - 7577 MiB free
llama_model_loader: loaded meta data with 38 key-value pairs and 339 tensors from /home/prometheus/.cache/huggingface/hub/models--bartowski--Qwen2.5-7B-Instruct-GGUF/snapshots/8911e8a47f92bac19d6f5c64a2e2095bd2f7d031/./Qwen2.5-7B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 7B Instruct
llama_model_loader: - kv   3:

In [16]:
prompt_depth1 = requests.get(BASE_URL+'prompts/extraction_depth1').text[:-42]
prompt_other = requests.get(BASE_URL+'prompts/extraction_other').text

In [41]:
with open(BASE_FOLDER + 'prompts/extraction_depth1') as f:
    prompt_depth1 = f.read()
#print(prompt_depth1)
with open(BASE_FOLDER + 'prompts/extraction_other') as f:
    prompt_other = f.read()
#print(prompt_other.format(goal='hello'))

prompt_reflection = """Check and refine your results based on the following criteria :
(1) Are there any mechanics that are not DIRECTLY related to the goal ?
Respond a complete result in the same format as before .
"""
"""
    Check and refine your results based on the following criteria :
(1) Are there any mechanics that are not **DIRECTLY** related to "{goal}"? 
(2) Are there any mechanics that are related to "{goal}" but depend from other already extracted?
If any mechanic contribute but not directly to the goal or depend on other extracted in this step remove it from your response
Respond a complete result in the same format as before and with nothing else.
"""
"""
EXAMPLES:
correct reflection: since drawing a card does not contribute directly to the scores, but is only one step to claim routes (the action that directly contribute to the score), should be removed
wrong reflection: dwawing cards directly contributes to the goal by providing the necessary resources to claim routes and complete tickets.

correct reflection: Increasing the number of action does not directly change the amount of shields, instead depends on other mechanics to do so, for this reason will be discarded
wrong reflection: Increasing the number of Actions available allows you to play more cards, including Action cards that can gain more <shield> or buy better cards."""
print(prompt_reflection.format(goal='GOAL'))


    Check and refine your results based on the following criteria :
(1) Are there any mechanics that are not **DIRECTLY** related to "GOAL"? 
(2) Are there any mechanics that are related to "GOAL" but depend from other already extracted?
If any mechanic contribute but not directly to the goal or depend on other extracted in this step remove it from your response
Respond a complete result in the same format as before and with nothing else.



In [44]:
to_do = [g for g in FILE_NAMES if f'{g}_local.json' not in os.listdir(OUT_FOLDER)
                                 or f'{g}_local.json' in OVERWRITE]
REFLECTION = False
if to_do == []:
    print('Nothing to do')
MAX_EXTRACTION_DEPTH = 6
for g in to_do:
    depth = {}
    rulebook = requests.get(BASE_URL +'rules/texts/'+g+'.txt').text
    name = g.replace('_',' ')
    messages = generate_message( prompt_depth1 , f"\nHere is the full rulebook of the game {name}:\n"+rulebook)
    out = model.create_chat_completion(messages,temperature=models[CHOSEN]['temperature'])['choices'][0]['message']
    messages.append(out)
    out = out['content']
    if(VERBOSE): print(f'pre-reflection: {out}')
    if(REFLECTION):
        messages.append({
            "role": "user",
            "content": prompt_reflection.format(goal='the end of the game')
        })
        reflection = model.create_chat_completion(messages,temperature=models[CHOSEN]['temperature'])['choices'][0]['message']
        messages.append(reflection)
        out = reflection['content']
        if(VERBOSE): print(f'after reflection: {out}')
    out = f'{{"": { out[8:-4] } }}'
        
    depth['1'] = [e['name'] for e in json.loads(out)['']]
    if(VERBOSE): print(depth['1'])
    for d in range(2,MAX_EXTRACTION_DEPTH):
        depth[str(d)] = []
        for e in depth[str(d-1)]:
            if(VERBOSE): print(f'descending from: {e}')
            messages.append({
                    "role": "user",
                    "content": prompt_other.format(goal=e)
                })
            #print(prompt_other.format(goal=e))
            out = model.create_chat_completion(messages,temperature=models[CHOSEN]['temperature'])['choices'][0]['message']
            messages.append(out)
            out = out['content']
            if(VERBOSE): print(f'pre-reflection: {out}')
            if(REFLECTION):
                messages.append({
                        "role": "user",
                        "content": prompt_reflection.format(goal=e)
                    })
                #print(prompt_reflection.format(goal=e))
                reflection = model.create_chat_completion(messages,temperature=models[CHOSEN]['temperature'])['choices'][0]['message']
                messages.append(reflection)
                out = reflection['content']
                if(VERBOSE): print(f'after reflection: {out}')
                
                
            out = f'{{"": { out[8:-4] } }}'
            
            depth[str(d)] += [e['name'] for e in json.loads(out)['']]
    with open(f'{OUT_FOLDER}/{g}_local.json', 'w') as f:
        json.dump(evaluation_error_output, f, indent=4)
        

Llama.generate: 432 prefix-match hit, remaining 1859 prompt tokens to eval
llama_perf_context_print:        load time =    6153.85 ms
llama_perf_context_print: prompt eval time =    4501.38 ms /  1859 tokens (    2.42 ms per token,   412.98 tokens per second)
llama_perf_context_print:        eval time =   52420.23 ms /   366 runs   (  143.22 ms per token,     6.98 tokens per second)
llama_perf_context_print:       total time =   57404.15 ms /  2225 tokens
llama_perf_context_print:    graphs reused =        363
Llama.generate: 2657 prefix-match hit, remaining 143 prompt tokens to eval


pre-reflection: ```json
[
	{
		"name": "Claim 1 Route",
		"type": "Contribute",
		"description": "Claim a route by playing train cards that match the route's color (or gray route with any color). Place a train on each space and discard the used cards.",
		"reasoning": "Claiming routes directly adds points to the player's score based on the length of the route claimed."
	},
	{
		"name": "Draw Train Cards",
		"type": "Contribute",
		"description": "Draw up to 2 train cards from the face-up pile or the deck.",
		"reasoning": "Drawing more cards provides more opportunities to claim routes or complete tickets, which contributes to the goal."
	},
	{
		"name": "Draw Tickets",
		"type": "Contribute",
		"description": "Draw 3 tickets and keep at least 1, adding potentially valuable objectives to complete.",
		"reasoning": "Drawing tickets increases the chance of completing objectives that can add significant points to the player's score."
	},
	{
		"name": "Create Longest Continuous Path",
		"ty

llama_perf_context_print:        load time =    6153.85 ms
llama_perf_context_print: prompt eval time =     579.58 ms /   143 tokens (    4.05 ms per token,   246.73 tokens per second)
llama_perf_context_print:        eval time =   51167.13 ms /   337 runs   (  151.83 ms per token,     6.59 tokens per second)
llama_perf_context_print:       total time =   52177.65 ms /   480 tokens
llama_perf_context_print:    graphs reused =        334
Llama.generate: 3137 prefix-match hit, remaining 141 prompt tokens to eval


pre-reflection: ```json
[
	{
		"name": "Draw Train Cards",
		"type": "Contribute",
		"description": "Draw up to 2 train cards from the face-up pile or the deck.",
		"reasoning": "Drawing train cards provides the necessary resources (cards) to claim a route."
	},
	{
		"name": "Have Train Cards in Hand",
		"type": "Contribute",
		"description": "Hold a sufficient number of train cards that match the color of the route.",
		"reasoning": "Having the required train cards allows you to claim a route."
	},
	{
		"name": "Face-Up Train Cards",
		"type": "Mixed",
		"description": "The top 5 face-up train cards in the deck can be used or replaced if 3 of them are locomotives.",
		"reasoning": "Face-up cards provide immediate access to resources, and the replacement rule adds variability."
	},
	{
		"name": "Train Deck",
		"type": "Contribute",
		"description": "Draw from the deck if the face-up cards do not contain the needed resources.",
		"reasoning": "The deck provides additional resources need

llama_perf_context_print:        load time =    6153.85 ms
llama_perf_context_print: prompt eval time =     596.26 ms /   141 tokens (    4.23 ms per token,   236.47 tokens per second)
llama_perf_context_print:        eval time =   42649.95 ms /   249 runs   (  171.28 ms per token,     5.84 tokens per second)
llama_perf_context_print:       total time =   43549.87 ms /   390 tokens
llama_perf_context_print:    graphs reused =        247
Llama.generate: 3527 prefix-match hit, remaining 139 prompt tokens to eval


...

llama_perf_context_print:        load time =    6153.85 ms
llama_perf_context_print: prompt eval time =     977.56 ms /   141 tokens (    6.93 ms per token,   144.24 tokens per second)
llama_perf_context_print:        eval time =   96012.71 ms /   226 runs   (  424.84 ms per token,     2.35 tokens per second)
llama_perf_context_print:       total time =   97265.68 ms /   367 tokens
llama_perf_context_print:    graphs reused =        224
Llama.generate: 16439 prefix-match hit, remaining 143 prompt tokens to eval


pre-reflection: ```json
[
	{
		"name": "Train Deck",
		"type": "Contribute",
		"description": "Shuffle the train cards and deal a starting hand of 4 cards to each player. The remaining deck is used for drawing cards.",
		"reasoning": "The train deck provides the cards that can be drawn."
	},
	{
		"name": "Face-Up Train Cards",
		"type": "Contribute",
		"description": "Turn the top 5 cards from the deck face up. If at least 3 of the 5 face-up train cards are locomotives, all 5 are discarded and replaced.",
		"reasoning": "The face-up cards provide immediate access to train cards that can be drawn."
	},
	{
		"name": "Draw Train Cards",
		"type": "Contribute",
		"description": "Draw up to 2 train cards from the face-up pile or the deck.",
		"reasoning": "Directly drawing train cards from the face-up pile or the deck."
	}
]
```
descending from: Discard and Replace


llama_perf_context_print:        load time =    6153.85 ms
llama_perf_context_print: prompt eval time =    1007.78 ms /   143 tokens (    7.05 ms per token,   141.90 tokens per second)
llama_perf_context_print:        eval time =   62150.36 ms /   144 runs   (  431.60 ms per token,     2.32 tokens per second)
llama_perf_context_print:       total time =   63327.24 ms /   287 tokens
llama_perf_context_print:    graphs reused =        142


pre-reflection: ```json
[
	{
		"name": "Face-Up Train Cards",
		"type": "Contribute",
		"description": "Turn the top 5 cards from the deck face up. If at least 3 of the 5 face-up train cards are locomotives, all 5 are discarded and replaced.",
		"reasoning": "This mechanic ensures a mix of card types in the face-up pile, adding strategy and variability."
	}
]
```

This is the direct mechanic that triggers and stops the "Discard and Replace" action. If there are no more mechanics or concepts to extract, the response would be an empty JSON array as shown below:

```json
[]
```


JSONDecodeError: Expecting ',' delimiter: line 9 column 1 (char 343)

In [ ]:
#print(depth1)
for n in [e['name'] for e in depth1['']]:
    print(n)

In [23]:
print(out)

{'role': 'assistant', 'content': '```json\n[\n\t{\n\t\t"name": "Draw Train Cards",\n\t\t"type": "Contribute",\n\t\t"description": "Draw up to 2 train cards per turn, either face up or from the deck.",\n\t\t"reasoning": "Drawing train cards provides the necessary resources to claim routes."\n\t},\n\t{\n\t\t"name": "Longest Continuous Path",\n\t\t"type": "Contribute",\n\t\t"description": "Create the longest continuous path of plastic trains of the same color to earn bonus points.",\n\t\t"reasoning": "Creating a longer path can indirectly contribute to the ability to claim more routes by having more trains available."\n\t},\n\t{\n\t\t"name": "Complete Tickets",\n\t\t"type": "Contribute",\n\t\t"description": "Complete the tickets you keep to earn points.",\n\t\t"reasoning": "Completing tickets can provide additional resources or strategies that help in claiming routes."\n\t}\n]\n```'}
